# 恢复动作幅度与标签：固定两种既有定义
只做同状态、同下一开盘重置目标的局部对照；不重训、不新建策略账户、不以均值选择beta。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.workflow.intraday_recovery_direct_allocation import project_legal_weights
from src.analysis.tiered_action_labels import reset_advantage
from src.backtest.ibkr_tiered import TieredFees
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/recovery_action_size_labels_20261014';plan=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/full_path_tiered_allocation_20261013';qlib.init(provider_uri=str(provider),region='us')
accounts,audit=_source(root,plan['account_source'],['inputs/frame',*[f'analysis/native/cash_start_2023_current_anchor_{c}' for c in plan['costs']]])
action,aa=_source(root,plan['action_source'],['inputs/plan','controller/source_recovery_selection']);spec=action['inputs/plan']['recovery_spec'];selection=action['controller/source_recovery_selection'];assert spec['teacher_recovery_beta']==1.2 and selection['action_beta']==.6
frame=accounts['inputs/frame'].set_index('trade_date',drop=False);labels=[];counts=[]
for cost in plan['costs']:
 book=accounts[f'analysis/native/cash_start_2023_current_anchor_{cost}'];path=book['independent_replay'].reset_index(drop=True);assert not path.recovery_action.any()
 targets=book['targets'].set_index('trade_date');orders=book['orders'].copy();orders['stamp']=pd.to_datetime(orders.stamp);orders=orders.sort_values('stamp').reset_index(drop=True)
 fee=TieredFees(plan['tariff'],slippage_bps=cost);order_position=0;teacher_only=common_count=0
 for i,row in path.iterrows():
  market=frame.loc[row.trade_date];stamp=row.trade_date+pd.Timedelta(hours=10,minutes=32)
  while order_position<len(orders) and orders.iloc[order_position].stamp<=stamp:
   order=orders.iloc[order_position];fee.commit(int(order.raw_shares),stamp=order.stamp,cap_applied=bool(order.cap_applied));order_position+=1
  beta=3*market.soxl_weight+market.soxx_weight
  ready=pd.notna(market.early_ret_soxx) and row.portfolio_drawdown>=spec['recovery_eligibility_drawdown']
  eligible_small=ready and min(beta,.6)>row.current_beta_cap+1e-12
  eligible_large=ready and min(beta,1.2)>row.current_beta_cap+1e-12
  teacher_only+=int(eligible_large and not eligible_small)
  if not eligible_small:continue
  common_count+=1;record={'slippage_bps':cost,'trade_date':row.trade_date,'label_available_at':pd.NaT,'small_advantage':np.nan,'large_advantage':np.nan,'small_target_beta':min(beta,.6),'large_target_beta':min(beta,1.2)}
  if i+1<len(path):
   nxt=frame.loc[path.iloc[i+1].trade_date];reset=targets.loc[nxt.trade_date,['soxl_weight','soxx_weight','cash_weight']].to_numpy(float);q=row[['ending_soxl_quantity','ending_soxx_quantity']].to_numpy(float);cash=float(row.ending_cash);before=dict(fee.monthly_shares)
   for name,limit in [('small',.6),('large',1.2)]:
    target=project_legal_weights(market.soxl_weight,market.soxx_weight,min(beta,limit),minimum_cash_weight=spec['minimum_cash_weight'],maximum_soxl_weight=spec['maximum_soxl_weight'])
    record[name+'_advantage']=reset_advantage(q,cash,row=market,next_row=nxt,target=target,reset_target=reset,fees=fee,entry_slot='recovery')
   assert fee.monthly_shares==before
   record['label_available_at']=nxt.trade_date+pd.Timedelta(hours=9,minutes=49)
  labels.append(record)
 counts.append({'slippage_bps':cost,'common_eligible':common_count,'teacher_only_eligible':teacher_only,'original_recovery_orders':int(path.recovery_action.sum())})
labels=pd.DataFrame(labels);statistics=[]
for cost,part in labels.groupby('slippage_bps'):
 for year in ['all',*sorted(part.trade_date.dt.year.unique())]:
  group=part if year=='all' else part.loc[part.trade_date.dt.year.eq(year)];valid=group.dropna(subset=['small_advantage','large_advantage'])
  statistics.append({'slippage_bps':cost,'year':str(year),'eligible':len(group),'scored':len(valid),'sign_disagreement_fraction':float(valid.small_advantage.gt(0).ne(valid.large_advantage.gt(0)).mean()),'small_positive_fraction':float(valid.small_advantage.gt(0).mean()),'large_positive_fraction':float(valid.large_advantage.gt(0).mean()),'mean_small_advantage_bp':float(valid.small_advantage.mean()*1e4),'mean_large_advantage_bp':float(valid.large_advantage.mean()*1e4)})
summary={'experiment':card.name,'decision':'same_state_recovery_action_size_label_diagnostic','new_fits':0,'new_strategy_accounts':0,'new_orders':0,'objective_achieved':False,'local_counterfactual_labels':int(labels.small_advantage.notna().sum()*2),'initial_cash':100000,'fee_role':'Existing fixed IBKR Tiered projection, two independent 2/5bp reference states; monthly original-open share counters cloned per branch','scope':'Recovery10:32 to next-open09:47 common-reset local advantage. No full policy value, no original teacher-label reconstruction, no independent OOS claim. Next reset uses future reference target as an explicitly retrospective common endpoint, never a decision feature.','provider_uri':str(provider),'region':'us','as_of':plan['as_of'],'qlib_docs':'https://qlib.readthedocs.io/en/latest/component/recorder.html','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/labels','sources/*']}}
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,recorder_name='frozen_recovery_size_labels',uri=str(tracking)):
 rec=R.get_recorder();rec.log_params(new_fits=0,new_strategy_accounts=0,small_beta=.6,large_beta=1.2)
 rec.save_objects(artifact_path='inputs',plan=plan,source_audit=audit,action_audit=aa,recovery_spec=spec,selection=selection)
 rec.save_objects(artifact_path='analysis',labels=labels,statistics=pd.DataFrame(statistics),eligibility=pd.DataFrame(counts),summary=summary)
 rec.save_objects(artifact_path='sources',label_helper=(root/'src/analysis/tiered_action_labels.py').read_text())
 artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')))
summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking));atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json');pd.DataFrame(statistics).to_csv(card/'label_comparison.csv',index=False);pd.DataFrame(counts).to_csv(card/'eligibility.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']},sort_keys=False))
assert hashlib.sha256((root/'production/stage1_futu_sim_v1/runtime_release.json').read_bytes()).hexdigest()==plan['release_sha256'];print(pd.DataFrame(statistics).to_string(index=False));print(pd.DataFrame(counts).to_string(index=False))


# 恢复动作幅度：标签方向大体一致，金额与资格域不同

固定比较源teacher beta上限1.2与实际action beta上限0.6。0拟合、0新增完整策略账户、0实际订单。复用10万美元、IBKR Tiered、2/5bp各自独立状态，不搜索beta或概率门槛。

每个局部对照从保存的开盘后实际数量/现金出发，以当日10:32原始股价和研究价格分别结算恢复或HOLD，下一09:47用同一个参考目标重置两分支；佣金、外部费、滑点、整数原始股数与月度份额计数均沿用既有引擎，分支复制计数不污染原状态。下一目标是事后共同终点，绝不作为当日决策输入。本标签不是完整策略价值，也不是原始595条teacher标签的重放。

共同合格状态2bp 228日、5bp 476日，各末端1日缺下一共同重置，保持未知。成熟标签分别227/475日，两动作正负分歧均3日，即1.32%/0.63%。因此动作幅度差异没有大面积翻转这些状态中的二元方向，不能单凭teacher/actual上限不同就解释模型不触发。

幅度仍影响收益金额：2bp小/大动作平均局部优势-2.09/-4.10bp，5bp为+4.81/+14.17bp。2bp的2025年两者均50%正标签，但平均优势-25.17/-45.41bp；说明正确率或胜率不能代替期望净优势。两成本路径有不同持仓、风险状态和资格样本，不能将5bp正均值解释为提高成本能改善收益；这些均值也没有独立置信或晋升资格。

资格域另有差异：仅beta1.2能够超过当时cap，而beta0.6不能的日数为60/70。该现象描述训练参考动作可覆盖更宽状态域，不证明应扩大实际beta。

回归检查先发现本轮新增entry_slot参数未切换入场价格；该首次结果已作废，见superseded_attempt.json。修复后恢复点50、开盘点100的独立用例验证了正确时点和两分支费计数隔离，原开盘默认行为也通过。最终notebook已重新执行，只有当前summary/Recorder结果有效。

决定：same_state_recovery_action_size_label_diagnostic。不调beta、不降低0.825门槛、不改训练标签。下一步若继续核查，应检验冻结恢复评分对净优势金额和损失严重度的区分能力，不只看正负标签；保持不拟合，避免重复开盘动作层已有实验。当前没有新的完整策略收益或稳定超额证据，生产release未改。

[按年及成本标签对照](label_comparison.csv) · [资格域差异](eligibility.csv)

Recorder `3883915d770e45b29310a72760cbc508`。
